In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install google-genai

In [ ]:
import os
import sys
import time
from google import genai
from google.genai import types
from google.colab import userdata

# Retrieve the API key securely from Colab Secrets
try:
    api_key = userdata.get('GEMINI_API_KEY')
except Exception as e:
    print("Error: Make sure you added 'GEMINI_API_KEY' to your Colab Secrets (key icon on the left).")
    raise e

# Initialize the Gemini Client
client = genai.Client(api_key=api_key)

# Define the file path in Google Drive where logs will be saved
LOG_FILE_PATH = "/content/drive/MyDrive/security_agent_log.txt"

# Helper function to write logs to your Google Drive
def log_to_drive(message: str):
    print(message)  # Still print to Colab console
    try:
        with open(LOG_FILE_PATH, "a") as f:
            f.write(message + "\n")
    except IOError:
        print(f"Warning: Could not write to Google Drive path: {LOG_FILE_PATH}")

# =====================================================================
# 1. UPGRADED SIMULATED SCADA ENVIRONMENT (Defense-in-Depth)
# =====================================================================

class MultiVectorSCADATank:
    def __init__(self):
        # Base physical metrics
        self.temperature = 65.0
        self.cooling_pump = "NORMAL"  # NORMAL, OFF, EMERGENCY_OVERRIDE

        # Cyber state flags
        self.is_patched = False               # Fixes the API pump override
        self.credentials_compromised = True   # Attacker successfully brute-forced admin pass
        self.firmware_tampered = True         # Attacker uploaded a backdoored image
        self.blocked_ips = set()

        # Expanded logs featuring multiple attack vectors
        self.network_logs = [
            # Event 1: The API pump override exploit
            {"ip": "10.0.45.199", "payload": "POST /api/v1/control/pump_override?state=OFF", "status": "200 OK"},
            # Event 2: Brute force login success from a different malicious IP
            {"ip": "172.16.8.55", "payload": "POST /api/v1/auth/login with password=admin", "status": "200 OK - SESSION CREATED"},
            # Event 3: Unauthorized firmware payload upload from a third IP
            {"ip": "198.51.100.4", "payload": "PUT /api/v1/system/firmware_update payload=0xBADF15M", "status": "200 OK"}
        ]

    def update_environment_state(self):
        """Simulates physical progress over time based on current cyber states."""

        # 1. Check if the pump is disabled via the unpatched API exploit
        pump_disabled_by_api = False
        for log in self.network_logs:
            if log["ip"] not in self.blocked_ips:
                if "pump_override?state=OFF" in log["payload"] and not self.is_patched:
                    pump_disabled_by_api = True

        # 2. Check if compromised credentials are still active (causes temperature to drift upward)
        credential_drift = 0.0
        if self.credentials_compromised:
            # Simulated attacker changes setpoints using the compromised account
            credential_drift = 5.0

        # 3. Check if firmware is still tampered (injects noise, preventing normal pump cooling)
        firmware_impact = 0.0
        if self.firmware_tampered:
            # Backdoor firmware randomly cuts cooling efficiency
            firmware_impact = 10.0

        # Calculate final state
        if pump_disabled_by_api:
            self.cooling_pump = "OFF"

        if self.cooling_pump == "OFF":
            self.temperature += (15.0 + credential_drift + firmware_impact)
        elif self.cooling_pump == "EMERGENCY_OVERRIDE":
            self.temperature = max(50.0, self.temperature - 20.0)
        else:
            # Normal operations with active threat penalties
            self.temperature += (credential_drift + firmware_impact)
            # Normal passive cooling decay
            if self.temperature > 65.0:
                self.temperature -= 3.0

# Initialize the mock physical asset
tank_system = MultiVectorSCADATank()

# =====================================================================
# 2. DEFENDER AGENT TOOLS (With 12-second throttle to respect Free Tier API)
# =====================================================================

def query_scada_telemetry() -> dict:
    """
    Returns the current operational metrics of the SCADA tank,
    including current temperature, cooling status, firmware integrity status,
    credential state, and recent network logs.
    Use this to monitor system health and detect multi-vector anomalies.
    """
    log_to_drive("[TOOL RUNNING] query_scada_telemetry()")
    time.sleep(12)

    tank_system.update_environment_state()
    return {
        "current_temperature_celsius": tank_system.temperature,
        "cooling_pump_status": tank_system.cooling_pump,
        "firewall_blocked_ips": list(tank_system.blocked_ips),
        "system_patched": tank_system.is_patched,
        "admin_credentials_compromised": tank_system.credentials_compromised,
        "firmware_integrity_compromised": tank_system.firmware_tampered,
        "recent_network_logs": tank_system.network_logs
    }

def apply_network_block(ip_address: str) -> str:
    """
    Blocks a specific IP address on the industrial firewall to stop active connections.
    Args:
        ip_address: The IP address to block.
    """
    log_to_drive(f"[TOOL RUNNING] apply_network_block('{ip_address}')")
    time.sleep(12)
    tank_system.blocked_ips.add(ip_address)
    return f"Success: IP address {ip_address} has been blacklisted on the firewall."

def install_security_patch() -> str:
    """
    Deploys a software patch to the SCADA API. This prevents unauthorized
    override commands from modifying the cooling pump state in the future.
    """
    log_to_drive("[TOOL RUNNING] install_security_patch()")
    time.sleep(12)
    tank_system.is_patched = True
    return "Success: Security patch successfully applied to SCADA API controller."

def trigger_emergency_cooling() -> str:
    """
    Forces the cooling system into EMERGENCY_OVERRIDE mode. Use this if
    the system temperature is dangerously high (>= 90.0C).
    """
    log_to_drive("[TOOL RUNNING] trigger_emergency_cooling()")
    time.sleep(12)
    tank_system.cooling_pump = "EMERGENCY_OVERRIDE"
    return "Success: Emergency override cooling has been manually engaged."

def rotate_admin_credentials() -> str:
    """
    Forces a rotation of all administrator credentials. This instantly logs out
    unauthorized attackers using brute-forced passwords and restores 'admin_credentials_compromised' to False.
    """
    log_to_drive("[TOOL RUNNING] rotate_admin_credentials()")
    time.sleep(12)
    tank_system.credentials_compromised = False
    return "Success: Admin credentials rotated. All unauthorized active sessions terminated."

def reflash_secure_firmware() -> str:
    """
    Reflashes the SCADA controller's flash memory with a cryptographically verified
    original firmware image. This removes backdoors and restores 'firmware_integrity_compromised' to False.
    """
    log_to_drive("[TOOL RUNNING] reflash_secure_firmware()")
    time.sleep(12)
    tank_system.firmware_tampered = False
    return "Success: Secure golden firmware re-imaged. Rootkits and backdoors removed."

# =====================================================================
# 3. AGENTIC EXECUTION
# =====================================================================

def run_cyber_defense_agent():
    # Clear previous logs in Drive if they exist
    try:
        with open(LOG_FILE_PATH, "w") as f:
            f.write("=== UPGRADED SCADA DEFENSE LOG ===\n")
    except IOError:
        pass

    model_name = "gemini-2.5-flash"

    system_instruction = (
        "You are an autonomous Cyber-Physical Security Incident Response Agent.\n"
        "Your task is to defend a critical industrial chemical storage tank.\n"
        "Your objectives:\n"
        "1. Maintain the system temperature below the critical safety threshold of 90.0 degrees Celsius.\n"
        "2. Analyze telemetry and logs to detect ALL active vulnerabilities and malicious activities.\n"
        "3. Mitigate immediate thermal/physical threats first using emergency cooling if temperature >= 90.0.\n"
        "4. Systematically eliminate all cyber compromises: block attacking IPs, rotate "
        "compromised credentials, reflash backdoored firmware, and patch application flaws.\n"
        "5. Verify system safety. Do not finish until all threat vectors are remediated and the system has stabilized."
    )

    tools = [
        query_scada_telemetry,
        apply_network_block,
        install_security_patch,
        trigger_emergency_cooling,
        rotate_admin_credentials,
        reflash_secure_firmware
    ]

    user_goal = (
        "Perform a full forensic audit. Identify all active compromises, execute "
        "remediation actions for all physical and cyber threat vectors, ensure the "
        "temperature is stabilized safely, and produce a unified incident response summary."
    )

    log_to_drive("=== Launching Defense-in-Depth Cyber Security Agent ===")

    response = client.models.generate_content(
        model=model_name,
        contents=user_goal,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            tools=tools,
            temperature=0.1,
        )
    )

    log_to_drive("\n=== Agent Final Comprehensive Forensics & Incident Report ===")
    log_to_drive(response.text)
    log_to_drive("\n=== LOG END ===")

# Run the simulation
run_cyber_defense_agent()

=== Launching Autonomous CPS Defense Agent ===
[TOOL RUNNING] query_scada_telemetry()
[TOOL RUNNING] apply_network_block('10.0.45.199')
[TOOL RUNNING] install_security_patch()
[TOOL RUNNING] query_scada_telemetry()
[TOOL RUNNING] trigger_emergency_cooling()
[TOOL RUNNING] query_scada_telemetry()


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 34.718084912s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'

In [ ]:
import os
import sys
import time
from google import genai
from google.genai import types
from google.colab import userdata

# Retrieve the API key securely from Colab Secrets
try:
    api_key = userdata.get('GEMINI_API_KEY')
except Exception as e:
    print("Error: Make sure you added 'GEMINI_API_KEY' to your Colab Secrets (key icon on the left).")
    raise e

# Initialize the Gemini Client
client = genai.Client(api_key=api_key)

# Define the file path in Google Drive where logs will be saved
LOG_FILE_PATH = "/content/drive/MyDrive/security_agent_log.txt"

# Helper function to write logs to your Google Drive
def log_to_drive(message: str):
    print(message)  # Still print to Colab console
    try:
        with open(LOG_FILE_PATH, "a") as f:
            f.write(message + "\n")
    except IOError:
        print(f"Warning: Could not write to Google Drive path: {LOG_FILE_PATH}")

# =====================================================================
# 1. UPGRADED SIMULATED SCADA ENVIRONMENT (Defense-in-Depth)
# =====================================================================

class MultiVectorSCADATank:
    def __init__(self):
        # Base physical metrics
        self.temperature = 65.0
        self.cooling_pump = "NORMAL"  # NORMAL, OFF, EMERGENCY_OVERRIDE

        # Cyber state flags
        self.is_patched = False               # Fixes the API pump override
        self.credentials_compromised = True   # Attacker successfully brute-forced admin pass
        self.firmware_tampered = True         # Attacker uploaded a backdoored image
        self.blocked_ips = set()

        # Expanded logs featuring multiple attack vectors
        self.network_logs = [
            # Event 1: The API pump override exploit
            {"ip": "10.0.45.199", "payload": "POST /api/v1/control/pump_override?state=OFF", "status": "200 OK"},
            # Event 2: Brute force login success from a different malicious IP
            {"ip": "172.16.8.55", "payload": "POST /api/v1/auth/login with password=admin", "status": "200 OK - SESSION CREATED"},
            # Event 3: Unauthorized firmware payload upload from a third IP
            {"ip": "198.51.100.4", "payload": "PUT /api/v1/system/firmware_update payload=0xBADF15M", "status": "200 OK"}
        ]

    def update_environment_state(self):
        """Simulates physical progress over time based on current cyber states."""

        # 1. Check if the pump is disabled via the unpatched API exploit
        pump_disabled_by_api = False
        for log in self.network_logs:
            if log["ip"] not in self.blocked_ips:
                if "pump_override?state=OFF" in log["payload"] and not self.is_patched:
                    pump_disabled_by_api = True

        # 2. Check if compromised credentials are still active (causes temperature to drift upward)
        credential_drift = 0.0
        if self.credentials_compromised:
            # Simulated attacker changes setpoints using the compromised account
            credential_drift = 5.0

        # 3. Check if firmware is still tampered (injects noise, preventing normal pump cooling)
        firmware_impact = 0.0
        if self.firmware_tampered:
            # Backdoor firmware randomly cuts cooling efficiency
            firmware_impact = 10.0

        # Calculate final state
        if pump_disabled_by_api:
            self.cooling_pump = "OFF"

        if self.cooling_pump == "OFF":
            self.temperature += (15.0 + credential_drift + firmware_impact)
        elif self.cooling_pump == "EMERGENCY_OVERRIDE":
            self.temperature = max(50.0, self.temperature - 20.0)
        else:
            # Normal operations with active threat penalties
            self.temperature += (credential_drift + firmware_impact)
            # Normal passive cooling decay
            if self.temperature > 65.0:
                self.temperature -= 3.0

# Initialize the mock physical asset
tank_system = MultiVectorSCADATank()

# =====================================================================
# 2. DEFENDER AGENT TOOLS (With 12-second throttle to respect Free Tier API)
# =====================================================================

def query_scada_telemetry() -> dict:
    """
    Returns the current operational metrics of the SCADA tank,
    including current temperature, cooling status, firmware integrity status,
    credential state, and recent network logs.
    Use this to monitor system health and detect multi-vector anomalies.
    """
    log_to_drive("[TOOL RUNNING] query_scada_telemetry()")
    time.sleep(12)

    tank_system.update_environment_state()
    return {
        "current_temperature_celsius": tank_system.temperature,
        "cooling_pump_status": tank_system.cooling_pump,
        "firewall_blocked_ips": list(tank_system.blocked_ips),
        "system_patched": tank_system.is_patched,
        "admin_credentials_compromised": tank_system.credentials_compromised,
        "firmware_integrity_compromised": tank_system.firmware_tampered,
        "recent_network_logs": tank_system.network_logs
    }

def apply_network_block(ip_address: str) -> str:
    """
    Blocks a specific IP address on the industrial firewall to stop active connections.
    Args:
        ip_address: The IP address to block.
    """
    log_to_drive(f"[TOOL RUNNING] apply_network_block('{ip_address}')")
    time.sleep(12)
    tank_system.blocked_ips.add(ip_address)
    return f"Success: IP address {ip_address} has been blacklisted on the firewall."

def install_security_patch() -> str:
    """
    Deploys a software patch to the SCADA API. This prevents unauthorized
    override commands from modifying the cooling pump state in the future.
    """
    log_to_drive("[TOOL RUNNING] install_security_patch()")
    time.sleep(12)
    tank_system.is_patched = True
    return "Success: Security patch successfully applied to SCADA API controller."

def trigger_emergency_cooling() -> str:
    """
    Forces the cooling system into EMERGENCY_OVERRIDE mode. Use this if
    the system temperature is dangerously high (>= 90.0C).
    """
    log_to_drive("[TOOL RUNNING] trigger_emergency_cooling()")
    time.sleep(12)
    tank_system.cooling_pump = "EMERGENCY_OVERRIDE"
    return "Success: Emergency override cooling has been manually engaged."

def rotate_admin_credentials() -> str:
    """
    Forces a rotation of all administrator credentials. This instantly logs out
    unauthorized attackers using brute-forced passwords and restores 'admin_credentials_compromised' to False.
    """
    log_to_drive("[TOOL RUNNING] rotate_admin_credentials()")
    time.sleep(12)
    tank_system.credentials_compromised = False
    return "Success: Admin credentials rotated. All unauthorized active sessions terminated."

def reflash_secure_firmware() -> str:
    """
    Reflashes the SCADA controller's flash memory with a cryptographically verified
    original firmware image. This removes backdoors and restores 'firmware_integrity_compromised' to False.
    """
    log_to_drive("[TOOL RUNNING] reflash_secure_firmware()")
    time.sleep(12)
    tank_system.firmware_tampered = False
    return "Success: Secure golden firmware re-imaged. Rootkits and backdoors removed."

# =====================================================================
# 3. AGENTIC EXECUTION
# =====================================================================

def run_cyber_defense_agent():
    # Clear previous logs in Drive if they exist
    try:
        with open(LOG_FILE_PATH, "w") as f:
            f.write("=== UPGRADED SCADA DEFENSE LOG ===\n")
    except IOError:
        pass

    model_name = "gemini-2.5-flash"

    system_instruction = (
        "You are an autonomous Cyber-Physical Security Incident Response Agent.\n"
        "Your task is to defend a critical industrial chemical storage tank.\n"
        "Your objectives:\n"
        "1. Maintain the system temperature below the critical safety threshold of 90.0 degrees Celsius.\n"
        "2. Analyze telemetry and logs to detect ALL active vulnerabilities and malicious activities.\n"
        "3. Mitigate immediate thermal/physical threats first using emergency cooling if temperature >= 90.0.\n"
        "4. Systematically eliminate all cyber compromises: block attacking IPs, rotate "
        "compromised credentials, reflash backdoored firmware, and patch application flaws.\n"
        "5. Verify system safety. Do not finish until all threat vectors are remediated and the system has stabilized."
    )

    tools = [
        query_scada_telemetry,
        apply_network_block,
        install_security_patch,
        trigger_emergency_cooling,
        rotate_admin_credentials,
        reflash_secure_firmware
    ]

    user_goal = (
        "Perform a full forensic audit. Identify all active compromises, execute "
        "remediation actions for all physical and cyber threat vectors, ensure the "
        "temperature is stabilized safely, and produce a unified incident response summary."
    )

    log_to_drive("=== Launching Defense-in-Depth Cyber Security Agent ===")

    response = client.models.generate_content(
        model=model_name,
        contents=user_goal,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            tools=tools,
            temperature=0.1,
        )
    )

    log_to_drive("\n=== Agent Final Comprehensive Forensics & Incident Report ===")
    log_to_drive(response.text)
    log_to_drive("\n=== LOG END ===")

# Run the simulation
run_cyber_defense_agent()

=== Launching Defense-in-Depth Cyber Security Agent ===
[TOOL RUNNING] query_scada_telemetry()
[TOOL RUNNING] trigger_emergency_cooling()
[TOOL RUNNING] reflash_secure_firmware()
[TOOL RUNNING] rotate_admin_credentials()
[TOOL RUNNING] install_security_patch()
[TOOL RUNNING] apply_network_block('10.0.45.199')
[TOOL RUNNING] apply_network_block('172.16.8.55')
[TOOL RUNNING] apply_network_block('198.51.100.4')
[TOOL RUNNING] query_scada_telemetry()

=== Agent Final Comprehensive Forensics & Incident Report ===
**Incident Response Summary**

**Initial Situation:**
A critical industrial chemical storage tank was found in a highly compromised state, posing significant physical and cyber security risks.
*   **Temperature:** The tank's temperature was critically high at 95°C, exceeding the safety threshold of 90°C.
*   **Cooling System:** The cooling pump was found to be deliberately turned OFF.
*   **Firmware Integrity:** The SCADA controller's firmware was compromised, indicating a potentia